# Customer Churn Prediction with an ANN

**Author:** Mohammed Alaa Eddine GUERBOUZ  
**University:** University of Ghardaia — Department of Mathematics and Computer Science  
**Academic period:** Master 2 SIEC · 2024–2025  
**Type:** Original university coursework, reorganized for a GitHub portfolio in 2026

**Problem:** Predict whether customers are likely to leave a service.

**Approach:** Dense neural network on tabular customer attributes.

> Historical coursework: the computational cells below are retained from the original submission. Results have not been independently rerun or revalidated as part of the portfolio reorganization. See this folder's README for data and execution notes.


---

<div class="alert alert-success">

<center><b>University of Ghardaia</b></center>
    
<center><i>Faculty of Science and Technology - Department of Mathematics and Computer Science – Master 2 SIEC - 2024/2025</i>
</center></div>

------

<br><br>

<center>
<span style='font-weight:bold;font-size:27px;'><span style='color:#5D00FF;'>TP 7  : Ann </span>
    </span>
    </center>
<br>


### GUERBOUZ MOHAMMED ALAA EDDINE

### Step 2: Import Libraries
We import the required libraries for data processing, model building, and evaluation.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import confusion_matrix
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical

### Step 3: Load and Preprocess the Data
We load the dataset, encode categorical variables, split the data into training and test sets, and normalize the features.

In [4]:
# Load the dataset
dataset = pd.read_csv('Churn_Modelling.csv')

# Extract features (X) and target (y)
X = dataset.iloc[:, 3:13].values  # Features: Columns 3 to 12
y = dataset.iloc[:, 13].values    # Target: Column 13 (Exited)

# Encode categorical data (Geography and Gender)
labelencoder_X_1 = LabelEncoder()
X[:, 1] = labelencoder_X_1.fit_transform(X[:, 1])  # Geography

labelencoder_X_2 = LabelEncoder()
X[:, 2] = labelencoder_X_2.fit_transform(X[:, 2])  # Gender

# One-hot encode the Geography column
ct = ColumnTransformer([("Geography", OneHotEncoder(), [1])], remainder='passthrough')
X = ct.fit_transform(X)
X = X[:, 1:]  # Remove one column to avoid dummy variable trap

# Split the dataset into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

# Feature scaling (normalization)
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

### Step 4: Build the Artificial Neural Network (ANN)
We create an ANN with two hidden layers, dropout for regularization, and an output layer with a sigmoid activation function.

In [5]:
# Initialize the ANN
classifier = Sequential()

# Add the input layer and the first hidden layer
classifier.add(Dense(units=6, kernel_initializer='uniform', activation='relu', input_dim=11))

# Add dropout for regularization
classifier.add(Dropout(rate=0.2))

# Add the second hidden layer
classifier.add(Dense(units=6, kernel_initializer='uniform', activation='relu'))

# Add dropout for regularization
classifier.add(Dropout(rate=0.2))

# Add the output layer
classifier.add(Dense(units=1, kernel_initializer='uniform', activation='sigmoid'))

# Compile the ANN
classifier.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the ANN on the training set
classifier.fit(X_train, y_train, batch_size=10, epochs=100)

Epoch 1/100
800/800 [==============================] - 1s 716us/step - loss: 0.4948 - accuracy: 0.7961
Epoch 2/100
800/800 [==============================] - 1s 715us/step - loss: 0.4412 - accuracy: 0.7960
Epoch 3/100
800/800 [==============================] - 1s 702us/step - loss: 0.4386 - accuracy: 0.7960
Epoch 4/100
800/800 [==============================] - 1s 701us/step - loss: 0.4364 - accuracy: 0.7960
Epoch 5/100
800/800 [==============================] - 1s 718us/step - loss: 0.4369 - accuracy: 0.7960
Epoch 6/100
800/800 [==============================] - 1s 715us/step - loss: 0.4387 - accuracy: 0.7960
Epoch 7/100
800/800 [==============================] - 1s 692us/step - loss: 0.4351 - accuracy: 0.7960
Epoch 8/100
800/800 [==============================] - 1s 716us/step - loss: 0.4390 - accuracy: 0.7960
Epoch 9/100
800/800 [==============================] - 1s 746us/step - loss: 0.4354 - accuracy: 0.7960
Epoch 10/100
800/800 [==============================] - 1s 709us/step - l

### Step 5: Make Predictions and Evaluate the Model
We use the trained model to predict on the test set and evaluate its performance using a confusion matrix and accuracy score.

In [9]:
# Predict the test set results
y_pred = classifier.predict(X_test)
y_pred = (y_pred > 0.5)  # Convert probabilities to binary predictions

# Create the confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

# Calculate accuracy
accuracy = (cm[0, 0] + cm[1, 1]) / (cm[0, 0] + cm[0, 1] + cm[1, 0] + cm[1, 1])
print(f"Accuracy: {accuracy * 100:.2f}%")

63/63 [==============================] - 0s 582us/step
Confusion Matrix:
[[1595    0]
 [ 405    0]]
Accuracy: 79.75%


### Step 6: Hyperparameter Tuning (Optional)
We use GridSearchCV to find the best hyperparameters for the ANN.

#### Manually Create the `KerasClassifier` Class
We create a custom `KerasClassifier` class to mimic the functionality of the original `KerasClassifier` wrapper.

In [12]:
from sklearn.base import BaseEstimator, ClassifierMixin
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

class KerasClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, build_fn=None, batch_size=10, epochs=100, optimizer='adam'):
        self.build_fn = build_fn
        self.batch_size = batch_size
        self.epochs = epochs
        self.optimizer = optimizer
        self.model = None

    def fit(self, X, y):
        # Build the model using the provided function
        self.model = self.build_fn(self.optimizer)
        
        # Train the model
        self.model.fit(X, y, batch_size=self.batch_size, epochs=self.epochs, verbose=0)
        return self

    def predict(self, X):
        # Predict using the trained model
        y_pred = self.model.predict(X)
        return (y_pred > 0.5).astype(int).flatten()

    def score(self, X, y):
        # Evaluate the model and return accuracy
        loss, accuracy = self.model.evaluate(X, y, verbose=0)
        return accuracy

#### Hyperparameter Tuning
We define a parameter grid and use `GridSearchCV` to find the best hyperparameters for the ANN.

In [13]:
from sklearn.model_selection import GridSearchCV

# Function to build the ANN
def build_classifier(optimizer='adam'):
    classifier = Sequential()
    classifier.add(Dense(units=6, kernel_initializer='uniform', activation='relu', input_dim=11))
    classifier.add(Dropout(rate=0.2))
    classifier.add(Dense(units=6, kernel_initializer='uniform', activation='relu'))
    classifier.add(Dropout(rate=0.2))
    classifier.add(Dense(units=1, kernel_initializer='uniform', activation='sigmoid'))
    classifier.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return classifier

# Create an instance of the custom KerasClassifier
classifier = KerasClassifier(build_fn=build_classifier)

# Define the parameter grid for hyperparameter tuning
parameters = {
    'batch_size': [10, 20],
    'epochs': [50, 100],
    'optimizer': ['adam', 'rmsprop']
}

# Perform grid search using GridSearchCV
grid_search = GridSearchCV(estimator=classifier, param_grid=parameters, scoring='accuracy', cv=3)
grid_search.fit(X_train, y_train)

# Print the best parameters and accuracy
best_parameters = grid_search.best_params_
best_accuracy = grid_search.best_score_
print(f"Best Parameters: {best_parameters}")
print(f"Best Accuracy: {best_accuracy * 100:.2f}%")

84/84 [==============================] - 0s 525us/step
Best Parameters: {'batch_size': 20, 'epochs': 50, 'optimizer': 'rmsprop'}
Best Accuracy: 81.68%
